### Initializations and imports
First of all we import the necessary libraries and modules.

In [15]:
import logging

import torch
import numpy as np
import random

from spikerplus.dataloaders import MnistDL
from spikerplus import Optimizer, NetBuilder, VhdlGenerator, Trainer
from spikerplus.vhdl import write_vhdl, compile_vhdl, elaborate_vhdl
from spikerplus.vhdl import NetworkSimulator

### Seed and reproducibility
To ensure **reproducibility** and **deterministic** execution, we fixed the random seed for Python, NumPy, and PyTorch. This is essential due to stochastic operations (e.g., Poisson-based spike encoding, random weight initialization).
Even though the **seed is set**, the results may still vary, especially when using different hardware or software environments ([PyTorch - Reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html?utm_source=chatgpt.com))

In [16]:
seed = 42
random.seed(seed)                 # Python built-in RNG
np.random.seed(seed)              # NumPy RNG
torch.manual_seed(seed)           # CPU RNG
torch.cuda.manual_seed(seed)      # GPU RNG (single GPU)
torch.cuda.manual_seed_all(seed)  # GPU RNG (all GPUs)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Parameters
We define the **parameters** for the network and the optimizer.
The **parameters** include the network configuration, training parameters andthe optimizer configuration.

In [17]:
net_config_best = {
    "n_cycles":             10,
    "n_inputs":             784,
    "layer_0": {
        "neuron_model":     "lif",
        "n_neurons":        75,
        "beta":             0.9375,
        "learn_beta":       False,
        "threshold":        1.0,
        "learn_threshold":  False,
        "reset_mechanism":  "subtract"
    },
    "layer_1": {
        "neuron_model":     "lif",
        "n_neurons":        10,
        "beta":             0.9375,
        "learn_beta":       False,
        "threshold":        1.0,
        "learn_threshold":  False,
        "reset_mechanism":  "none"
    }
}
optim_config = {
	"weights_bw"	: {
		"min"	: 6,
		"max"	: 6
	},
	"neurons_bw"	: {
		"min"	: 9,
		"max"	: 9
	},
	"fp_dec"	: {
		"min"	: 5,
		"max"	: 5
	}
}

TRAIN =             True
batch_size =        64
n_epochs =          25
data_dir =          "Mnist/data"
output_dir =        "output"
SD_PATH =           "./Trained/trained_state_dict.pt"

### Building dataloader and network
Before launching the training, we need to build the dataloader and the network.

To do these we use the Spiker+ classes:

- We load the MNIST dataset and convert it into spike trains using Poisson-based rate coding.
- The network is defined and instantiated using **Spiker+** (PyTorch-based SNN model):

In [18]:
logging.basicConfig(level=logging.INFO)
data_loader = MnistDL(data_dir = data_dir, num_steps = net_config_best["n_cycles"])
train_loader, test_loader = data_loader.load(batch_size = batch_size)
net_builder = NetBuilder(net_config_best)  # Create the network
snn = net_builder.build()           # Build snn model

INFO:root:Network configured: 
{
    "n_cycles": 10,
    "n_inputs": 784,
    "layer_0": {
        "n_neurons": 75,
        "neuron_model": "lif",
        "threshold": 1.0,
        "learn_threshold": false,
        "reset_mechanism": "subtract",
        "beta": 0.9375,
        "learn_beta": false
    },
    "layer_1": {
        "n_neurons": 10,
        "neuron_model": "lif",
        "threshold": 1.0,
        "learn_threshold": false,
        "reset_mechanism": "none",
        "beta": 0.9375,
        "learn_beta": false
    }
}

INFO:root:Network ready: SNN(
  (layers): ModuleDict(
    (fc1): Linear(in_features=784, out_features=75, bias=False)
    (lif1): Leaky()
    (fc2): Linear(in_features=75, out_features=10, bias=False)
    (lif2): Leaky()
  )
)



### Network training
We train the network using the Spiker+ Trainer class.
If `TRAIN=True`, the network undergoes supervised training, employing surrogate-gradient-based **Back-Propagation Through Time** (**BPTT**).

At the end of the training phase, if the `store=True` parameter is defined, the trained state dict is saved for later inference.

In [12]:
if (TRAIN):
    trainer = Trainer(snn)
    trainer.train(train_loader, test_loader, n_epochs = n_epochs, store=True)
else:
    state_dict = torch.load(SD_PATH, weights_only=True)
    snn.load_state_dict(state_dict)

INFO:root:Training set-up: 
Readout type: mem
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0
)
Loss function: CrossEntropyLoss()
Device: cpu

INFO:root:Epochs: 25
Batch size: 64
Training batches: 937
Training samples: 59968
Validation batches: 156
Validation samples: 9984

Begin training


INFO:root:Epoch 0
Elapsed time 31.60s
Train loss: 0.25
Validation loss: 0.27
Train accuracy: 90.68342447280884%
Validation accuracy: 94.68749761581421%

INFO:root:Epoch 1
Elapsed time 50.04s
Train loss: 0.30
Validation loss: 0.36
Train accuracy: 95.17895579338074%
Validation accuracy: 96.15927338600159%

INFO:root:Epoch 2
Elapsed time 81.90s
Train loss: 0.19
Validation loss: 0.29
Train accuracy: 96.44097089767456%
Validation accuracy: 96.86492085456848%

INFO:root:Epoch 3
Elapsed time 1

### Optimization and Post-training quantization

**Spiker+** performs quantization exploration for future hardware development. It explores the search space defined in the parameters and shows the user the accuracies obtained with them so that it's possible to choose the best configuration.

In this specific case the quantization parameters are set to:
- weights_bw = 6 bit
- neurons_bw = 9 bit
- fp_dec = 5 bit

that represent the configuration chosen for the MNIST Challenge.

In [13]:
opt = Optimizer(snn, net_config_best, optim_config)
_ = opt.optimize(test_loader)
optim_config = {}
optim_config["weights_bw"] 	= int(input("Pick the best weights bitwidth: "))
optim_config["neurons_bw"]	= int(input("Pick the best neurons bitwidth: "))
optim_config["fp_dec"]		= int(input("Pick the best number of fixed point digits: "))

INFO:root:Training set-up: 
Readout type: mem
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0
)
Loss function: CrossEntropyLoss()
Device: cpu

INFO:root:Network configured: 
{
    "n_cycles": 10,
    "n_inputs": 784,
    "layer_0": {
        "n_neurons": 75,
        "neuron_model": "lif",
        "threshold": 1.0,
        "learn_threshold": false,
        "reset_mechanism": "subtract",
        "beta": 0.9375,
        "learn_beta": false
    },
    "layer_1": {
        "n_neurons": 10,
        "neuron_model": "lif",
        "threshold": 1.0,
        "learn_threshold": false,
        "reset_mechanism": "none",
        "beta": 0.9375,
        "learn_beta": false
    }
}

INFO:root:Network configured: 
{
    "n_cycles": 10,
    "n_inputs": 784,
    "layer_0": {
        "n_neur

### VHDL generation
Post-training and optimization, the network is converted into synthesizable **VHDL**. **Spiker+** generates VHDL for FPGA deployment, tailored specifically to network quantization and hardware constraints. The output is structured for **FPGA** synthesis, including neuron models, memories, and interfaces.

In [14]:
vhdl_generator = VhdlGenerator(snn, optim_config)
vhdl_snn  = vhdl_generator.generate(functional=False, interface=True)
write_vhdl(vhdl_snn, rm=True,  output_dir = output_dir)